## QC scope and running instructions
Restart session and Run all after uploading the three Python files. MAX_BATCHES=None scans
all training samples only; use an integer for a shorter run. The ClassMix algorithm is unchanged.
PASS_TRAIN_SCAN means all training samples passed these data checks with at least one mix;
it does not certify model integration, augmentation quality, or improved DG accuracy.
Review skip rates, eligible donors and area statistics before changing donor selection.
Reports are written before plotting. Files under /content disappear when the runtime ends;
copy the report folder to Drive to retain them. Figures show the first successful mixes.


# Kuro augmentation QC — Colab
Preprocessing/normalization/D4 giữ nguyên. Không thêm lọc speckle.
Protocol mới là đề xuất cần Lead review. Notebook kiểm tra code, không chứng minh tăng accuracy.

Đặt repo trên Drive và chỉnh REPO_ROOT bên dưới, hoặc upload ba file `kuro_dataset.py`,
`kuro_augmentation.py`, `mixstyle.py` vào `/content/kuro_modules`.
Không cần tải weights. Raw được copy từ Drive sang Colab, không tạo dataset augmented.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, json, shutil, hashlib, time
REPO_ROOT = Path('/content/drive/MyDrive/final_cv')
MODULE_ROOT = Path('/content/kuro_modules')
RAW_ON_DRIVE = Path('/content/drive/MyDrive/kuro_siwo/raw/geobench_kuro_siwo.tortilla')
RAW_LOCAL = Path('/content/kuro_siwo/raw/geobench_kuro_siwo.tortilla')
REPORT_ROOT = Path('/content/kuro_augmentation_qc')
BATCH_SIZE = 8
MAX_BATCHES = None  # None: scan all training samples; integer: partial QC
MAX_EXAMPLES = 4
SEED = 42
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
 'numpy>=1.26,<3', 'pandas>=2.2,<4', 'pyarrow>=17,<27',
 'rasterio>=1.4,<2', 'matplotlib>=3.8,<4'], check=True)


In [ ]:
if not MODULE_ROOT.exists():
    MODULE_ROOT.mkdir(parents=True)
for name, relative in [('kuro_dataset.py','scripts/kuro_dataset.py'),
                       ('kuro_augmentation.py','scripts/kuro_augmentation.py'),
                       ('mixstyle.py','src/geoid/dg/mixstyle.py')]:
    source = REPO_ROOT / relative
    if not source.exists():
        source = Path('/content') / name
    if source.exists():
        shutil.copy2(source, MODULE_ROOT / name)
    assert (MODULE_ROOT / name).exists(), f'Missing {name}: upload or set REPO_ROOT'
sys.path.insert(0, str(MODULE_ROOT))
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import importlib
importlib.invalidate_caches()
for module_name in ['kuro_dataset','kuro_augmentation','mixstyle']:
    sys.modules.pop(module_name, None)
import kuro_dataset as kd
from kuro_augmentation import ClassMix
from mixstyle import MixStyle
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
report = {'status':'RUNNING', 'seed':SEED, 'real_model_integration':'NOT_TESTED',
          'experimental_validation':'NOT_TESTED', 'torch':str(torch.__version__),
          'module_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest()
                           for p in [MODULE_ROOT/name for name in ['kuro_dataset.py','kuro_augmentation.py','mixstyle.py']]}}


## Kiểm tra MixStyle độc lập — không cần model hoặc dữ liệu

In [ ]:
torch.manual_seed(SEED)
x = torch.randn(4, 8, 8, 8, requires_grad=True)
events = ['a', 'a', 'b', 'c']
ms = MixStyle(p=1)
y = ms(x, events)
assert y.shape == x.shape and torch.isfinite(y).all()
assert all(events[i] != events[j] for i,j in enumerate(ms.last_info['donor_indices']))
y.square().mean().backward()
assert torch.isfinite(x.grad).all() and x.grad.abs().sum() > 0
ms.eval()
state = torch.get_rng_state()
assert ms(x, events) is x and torch.equal(state, torch.get_rng_state())
ms.train()
assert ms(x, ['a']*4) is x
report['mixstyle_synthetic'] = 'PASS'
print('PASS: MixStyle độc lập; chưa kiểm tra encoder thật')


## Stage raw và tạo loader train — kiểm checksum bản phát hành

In [ ]:
assert BATCH_SIZE >= 2 and (MAX_BATCHES is None or MAX_BATCHES >= 1)
if not RAW_LOCAL.exists():
    assert RAW_ON_DRIVE.exists(), str(RAW_ON_DRIVE)
    RAW_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(RAW_LOCAL.parent).free < RAW_ON_DRIVE.stat().st_size:
        raise RuntimeError('Không đủ dung lượng Colab để stage raw')
    temporary = RAW_LOCAL.with_suffix('.partial')
    shutil.copyfile(RAW_ON_DRIVE, temporary)
    kd.verify_raw(temporary)
    temporary.replace(RAW_LOCAL)
archive = kd.RawArchive(RAW_LOCAL)  # verifies existing local file too
train = kd.KuroDataset(archive, 'train', augment=True)
loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                    generator=torch.Generator().manual_seed(SEED))
report['dataset_augmentation'] = train.augmentation_config
report['preprocess_config'] = kd.CONFIG
report['metadata_audit'] = archive.audit
print(archive.audit)


## ClassMix trên batch thật và kiểm tra chính xác nguồn từng pixel

In [ ]:
from collections import Counter
from itertools import islice
import math
assert MAX_BATCHES is None or (isinstance(MAX_BATCHES, int) and MAX_BATCHES > 0)
assert MAX_EXAMPLES >= 1
torch.manual_seed(SEED)
# Rebuild to make repeated scans reproducible, independently of previous cells.
loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                   generator=torch.Generator().manual_seed(SEED))
aug = ClassMix(p=0.5)
records, examples = [], []
source_classes, event_counts, batch_event_counts = Counter(), Counter(), Counter()
original_pixels, mixed_pixels = Counter(), Counter()
seen_ids = set()
seen = eligible = other_event_available = batches = 0
start = time.perf_counter()
iterator = loader if MAX_BATCHES is None else islice(loader, MAX_BATCHES)
for batch_number, original in enumerate(iterator):
    batches += 1
    n = len(original['event_id'])
    class_sets = [set(y.unique().tolist()) - {255} for y in original['mask']]
    batch_event_counts[len(set(original['event_id']))] += 1
    for i, classes in enumerate(class_sets):
        source_classes[len(classes)] += 1
        event_counts[original['event_id'][i]] += 1
        sample_id = original['sample_id'][i]
        assert sample_id not in seen_ids, f'Duplicate sample in scan: {sample_id}'
        seen_ids.add(sample_id)
        candidates = [j for j in range(n) if original['event_id'][j] != original['event_id'][i]]
        other_event_available += bool(candidates)
        eligible += any(len(class_sets[j]) >= 2 for j in candidates)
    mixed, info = aug(original, training=True)
    for r in info['records']:
        i = r['recipient_index']
        if r['reason'] == 'applied':
            j = r['donor_index']; m = info['mix_mask'][i]
            assert original['event_id'][i] != original['event_id'][j]
            assert torch.equal(m, torch.isin(original['mask'][j], original['mask'].new_tensor(r['classes'])))
            assert m.any() and not m.all()
            for key in ['sar','dem','mask','input_valid','valid_mask']:
                assert torch.equal(mixed[key][i], torch.where(m, original[key][j], original[key][i]))
            if len(examples) < MAX_EXAMPLES:
                keys = ['sar','dem','mask']
                examples.append({'donor':{k:original[k][j].clone() for k in keys},
                                 'recipient':{k:original[k][i].clone() for k in keys},
                                 'mixed':{k:mixed[k][i].clone() for k in keys},
                                 'mix_mask':m.clone(), 'record':dict(r)})
        else:
            for key in ['sar','dem','mask','input_valid','valid_mask']:
                assert torch.equal(mixed[key][i], original[key][i])
    assert torch.equal(mixed['valid_mask'], mixed['mask'] != 255)
    assert not (mixed['valid_mask'] & ~mixed['input_valid']).any()
    cnn, temporal = kd.model_inputs(mixed)
    assert cnn.shape == (n,7,224,224)
    assert temporal['S1GRD'].shape == (n,2,3,224,224)
    assert temporal['DEM'].shape == (n,1,3,224,224)
    assert all(torch.isfinite(v).all() for v in [cnn,*temporal.values()])
    assert aug(original, training=False)[0] is original
    for label in [0,1,2,255]:
        original_pixels[label] += int((original['mask'] == label).sum())
        mixed_pixels[label] += int((mixed['mask'] == label).sum())
    records.extend(info['records']); seen += n
    if batches % 25 == 0:
        print(f'Checked {seen}/{len(train)} training samples; batches={batches}')
counts = Counter(r['reason'] for r in records)
applied = counts['applied']
attempted = seen - counts['probability']
areas = torch.tensor([r['area_fraction'] for r in records if r['reason']=='applied'])
area_summary = None if not len(areas) else {
    'min':areas.min().item(), 'median':areas.median().item(), 'mean':areas.mean().item(),
    'max':areas.max().item(), 'below_1_percent':int((areas < .01).sum()),
    'above_99_percent':int((areas > .99).sum())}
report['classmix'] = {
    'p':aug.p, 'records':records, 'counts':dict(counts), 'samples_checked':seen,
    'training_samples_total':len(train), 'full_train_coverage':seen==len(train),
    'batches_checked':batches, 'classes_present_per_sample':dict(source_classes),
    'samples_per_event':dict(event_counts), 'distinct_events_per_batch':dict(batch_event_counts),
    'recipients_with_other_event_donor':other_event_available,
    'recipients_with_eligible_donor':eligible, 'attempted':attempted,
    'applied_rate_all_samples':applied/seen if seen else None,
    'applied_rate_attempted':applied/attempted if attempted else None,
    'mixed_area_fraction':area_summary,
    'pixel_counts_before':dict(original_pixels), 'pixel_counts_after':dict(mixed_pixels),
    'elapsed_seconds':time.perf_counter()-start}
report['status'] = ('INCOMPLETE_NO_MIX' if not applied else
                    'PASS_TRAIN_SCAN' if seen==len(train) else 'PASS_PARTIAL')
report['scope_note'] = 'Training data QC only; no real encoder, GPU or DG accuracy validation.'
(REPORT_ROOT/'qc_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps({k:v for k,v in report['classmix'].items() if k!='records'},indent=2))
print(report['status'], REPORT_ROOT/'qc_report.json')


## Hiển thị cả 6 kênh SAR, DEM và nhãn; mask ghép riêng

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import numpy as np
label_colors = ['#404040','#2878b5','#ff7f0e','#e6e6e6']
label_names = ['0 Background','1 Permanent water','2 Flood','255 Ignore']
cmap_labels = ListedColormap(label_colors)
norm_labels = BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)
for number, example in enumerate(examples):
    sources = [example[k] for k in ['donor','recipient','mixed']]
    r = example['record']
    fig, axes = plt.subplots(8,3,figsize=(12,24))
    for row in range(8):
        if row < 6:
            t,c = divmod(row,2)
            values = [s['sar'][t,c] for s in sources]
            label = f"SAR t={t}, {['VV','VH'][c]}"
        else:
            key = 'dem' if row==6 else 'mask'
            values = [s[key].squeeze() for s in sources]
            label = key
        low = min(v.min().item() for v in values)
        high = max(v.max().item() for v in values)
        for col,value in enumerate(values):
            if row==7:
                display = value.cpu().numpy().copy()
                display[display==255] = 3
                axes[row,col].imshow(display,cmap=cmap_labels,norm=norm_labels,interpolation='nearest')
            else:
                axes[row,col].imshow(value.cpu(),vmin=low,vmax=high,cmap='gray' if row<6 else 'terrain')
            axes[row,col].set_title(f"{label}: {['Donor A','Recipient B','Mixed'][col]}")
            axes[row,col].axis('off')
    fig.suptitle(f"A event={r['donor_event']} → B event={r['recipient_event']}; "
                 f"classes={r['classes']}; area={100*r['area_fraction']:.2f}%")
    fig.legend(handles=[Patch(color=c,label=n) for c,n in zip(label_colors,label_names)],
               loc='lower center',ncol=4)
    fig.tight_layout(rect=[0,.025,1,.98])
    fig.savefig(REPORT_ROOT/f'classmix_{number:02d}.png')
    plt.show(); plt.close(fig)
    fig, ax = plt.subplots()
    ax.imshow(example['mix_mask'].cpu(),cmap='gray',vmin=0,vmax=1,interpolation='nearest')
    ax.set_title(f"White = donor A; area={100*r['area_fraction']:.2f}%")
    fig.savefig(REPORT_ROOT/f'classmix_mask_{number:02d}.png')
    plt.show(); plt.close(fig)
if not examples:
    print('No successful mix to plot; inspect skip counts in qc_report.json.')
print('Examples are the first successful mixes, not a representative sample of all mixes.')


## Bàn giao training
- ERM: D4 như hiện tại, không ClassMix, không MixStyle.
- DG1: gọi `mixed, info = ClassMix(p=0.5)(batch, training=True)` trước `kd.model_inputs`.
- DG2: không ClassMix; đội model gắn `MixStyle(p=0.5, alpha=0.1)` vào encoder.
- `event_id` của mẫu ClassMix là ID recipient để indexing; dùng `classmix_provenance`
  để biết cả hai nguồn, không coi đó là mẫu thuần một event cho alignment loss.
- Validation/test không dùng ClassMix/D4, model gọi `.eval()` để tắt MixStyle.
- Giữ sampler, seed plan, budget nhất quán; module này không tự thay sampler.
- Notebook chưa kiểm tra ResNet/TerraMind thật, CUDA hay hiệu quả DG.
- Report chứa provenance và kết quả QC; copy thư mục report về Drive nếu muốn lưu.
